# Misc recipes

Small, self-contained snippets that come up often when driving the Istari Digital Platform from [`istari_fluent`](../fluent) but didn't earn their own notebook.

Each section is independent &mdash; you don't need to run them in order. They all assume you've connected to the platform once with `IstariPlatform.from_env()`; if that's new to you, see [Chaining jobs &middot; 1 &middot; Connect and verify](./chaining_jobs.ipynb).

## Setup

Reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` from `samples/.env` (same convention as the other recipes).

In [ ]:
from istari_fluent import IstariPlatform

platform = IstariPlatform.from_env()
print(platform)

## Resume from a saved job id

Jobs live on the platform independently of the notebook that submitted them. If you restart the kernel (or come back the next day) you don't have to re-submit &mdash; just look up the job by id and keep going.

`platform.get_job(id)` reconstructs the same `JobView` you had before &mdash; completed, with its product list cached. Paste a job id you printed earlier (e.g. `print(job.id)` in another notebook) into `RESUME_JOB_ID` below.

Analogous helpers exist for other entities: `platform.get_model(model_id)`, `platform.get_resource("Artifact", artifact_id)`.

In [ ]:
RESUME_JOB_ID: str = "d457ca89-5a58-472e-92f2-c70569e7b6ab"  # paste your saved job id here

job = platform.get_job(RESUME_JOB_ID)
print(f"Resumed: {job}")

## Archive models by filename

Bulk-archive every model whose stored filename matches a specific value. Useful
when a batch of files was uploaded with a bad name (e.g. the double-extension
`Group3-UAS-Requirements-v2.xlsx.xlsx`) and you need to retire them cleanly.

`platform.resources().type("model").filter(...)` is a lazy,
chainable `ResourceQuery` over `client.list_resources`. Iteration walks **all**
pages via the v2 client's `Page.iter_items()` (same mechanism as
`fluent/python-client-usage.md`); the query uses `size=100` by default to limit
round-trips.

The v2 API expects **list-valued** filters where OpenAPI defines an array, e.g.
`file_name=["report.xlsx"]` not a bare string.

Each match is a `ResourceSearchItem` (`name`, `id`, `archive_status`, ...). We
lift rows to `ModelView` with `platform.get_model(item.id)` only for archives.
`model.archive()` calls `client.archive_model(model_id)` &mdash; the operation
is recorded on the platform, so a dry-run preview is printed before any
destructive call.

In [ ]:
TARGET_FILENAME = "Group3-UAS-Requirements.xlsx"

# Resource list filters are list-typed in the SDK (Pydantic validates).
query = platform.resources().type("model").filter(file_name=[TARGET_FILENAME])

# Total without walking every item (single page-1, size=1 request → page.total).
print(f"Matched {query.count()} model(s) with file_name={TARGET_FILENAME!r}")

# Materialise with list(iter(query)) so CPython's list() doesn't call __len__
# first (an extra count round-trip). Same items you'd get from `for item in query:`.
targets = list(iter(query))
for item in targets:
    print(f"  id={item.id}  name={item.name!r}  archive_status={item.archive_status}")

for item in targets:
    model = platform.get_model(item.id)
    model.archive()
    print(f"Archived: {item.id}")

print("Done.")

## Example: systems query

`platform.systems()` is an `ItemQuery` bound to `list_systems`. Chain `.filter(archive_status="active")`, `.sort("-created")`, `.take(n)`, or iterate in a `for` loop &mdash; pagination is handled the same way as for resources.

In [ ]:
# Other factories return the same ItemQuery pattern: lazy until you iterate
# or call .first() / .take(n) / .count() / .all().

# Five most recently created systems (one line, minimal API usage).
for system in platform.systems().sort("-created").take(5):
    print(system.id, system.name)